# Paired PCB experiment — Colab A100

Run all cells in a fresh **A100** runtime. This notebook is intentionally thin: every scientific decision and artifact gate lives in repository code. Use the ready-to-run notebook copied by `pcb_defect.handoff`; its exact bundle SHA-256 and snapshot Git SHA are rendered automatically. Do not hand-edit them.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/pcb-defect-paired')
SOURCE_BUNDLE = DRIVE_ROOT / 'handoff' / 'pcb-defect-source.bundle'
SOURCE_BUNDLE_SHA256 = 'PASTE_FINAL_BUNDLE_SHA256'
EXPECTED_GIT_SHA = 'PASTE_FINAL_GIT_SHA'
REPO = Path('/content/pcb-defect-source')
DATASET = DRIVE_ROOT / 'dataset' / 'pcb'
WORKSPACE = DRIVE_ROOT / 'workspaces' / EXPECTED_GIT_SHA[:12]
BASE_MODEL = WORKSPACE / 'inputs' / 'base_model.pt'
if SOURCE_BUNDLE_SHA256.startswith('PASTE' + '_') or EXPECTED_GIT_SHA.startswith('PASTE' + '_'):
    raise RuntimeError('Paste the two immutable handoff values before Run all')
print({'bundle': str(SOURCE_BUNDLE), 'dataset': str(DATASET), 'workspace': str(WORKSPACE)})

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if sha256_file(SOURCE_BUNDLE) != SOURCE_BUNDLE_SHA256:
    raise RuntimeError('Source bundle SHA-256 mismatch')
if REPO.exists():
    observed = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()
    if observed != EXPECTED_GIT_SHA:
        raise RuntimeError('Existing /content checkout has the wrong Git SHA; restart runtime')
else:
    subprocess.run(['git', 'clone', str(SOURCE_BUNDLE), str(REPO)], check=True)
    subprocess.run(['git', 'checkout', '--detach', EXPECTED_GIT_SHA], cwd=REPO, check=True)
status = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, check=True, capture_output=True, text=True).stdout
if status:
    raise RuntimeError(f'Source checkout is dirty: {status}')
print('SOURCE GATE PASS', EXPECTED_GIT_SHA)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.11.18'], check=True)
UV = shutil.which('uv')
if not UV:
    raise RuntimeError('uv installation failed')
subprocess.run([UV, 'sync', '--locked', '--no-editable', '--reinstall-package', 'pcb-defect', '--extra', 'train', '--group', 'eval'], cwd=REPO, check=True)
VENV_PYTHON = REPO / '.venv' / 'bin' / 'python'
if not VENV_PYTHON.is_file():
    raise RuntimeError(f'Locked environment Python is missing: {VENV_PYTHON}')
def runtime_contract_state(label: str) -> dict[str, object]:
    command = [
        str(VENV_PYTHON),
        '-m',
        'pcb_defect.runtime_contract',
        '--require-cuda-provider',
    ]
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    print(f'[{label}] returncode={result.returncode}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'{label} FAILED')
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f'{label} returned no runtime state')
    try:
        return json.loads(lines[-1])
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'{label} returned invalid runtime JSON') from exc

LOCKED_RUNTIME_STATE = runtime_contract_state('LOCKED RUNTIME CONTRACT')
sys.path.insert(0, str(REPO / 'src'))
from pcb_defect.notebook_runtime import run_streaming_command
def import_probe(label: str, code: str) -> None:
    result = subprocess.run([str(VENV_PYTHON), '-c', code], cwd=REPO, text=True, capture_output=True)
    print(f'[{label}] returncode={result.returncode}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'IMPORT PROBE FAILED: {label}')
import_probe('python', "import sys, numpy; print(sys.version); print('numpy', numpy.__version__)")
import_probe('torch-cuda', "import torch; print('torch', torch.__version__); print('cuda', torch.version.cuda); print('available', torch.cuda.is_available()); assert torch.cuda.is_available(); name=torch.cuda.get_device_name(0); print('gpu', name); assert 'A100' in name")
import_probe('ultralytics', "import ultralytics; print('ultralytics', ultralytics.__version__)")
print('LOCKED ENVIRONMENT GATE PASS')

In [ ]:
if DATASET.exists() and not (DATASET / 'conversion_report.json').is_file():
    raise RuntimeError('Partial dataset directory exists; inspect it instead of overwriting')
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.data_prep.prepare', '--out', str(DATASET), '--strategy', 'grouped', '--seed', '42'], cwd=REPO, check=True)
subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.data_prep.paired', '--source', str(DATASET), '--config', str(REPO / 'configs/paired_protocol.yaml'), '--artifacts', str(REPO / 'reports/protocol'), '--runtime', str(WORKSPACE / 'runtime_data')], cwd=REPO, check=True)
print('DATA AND MANIFEST GATE PASS')

In [ ]:
def run_logged(label, command, log_path):
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(result.stdout + '\n--- STDERR ---\n' + result.stderr, encoding='utf-8')
    print(f'[{label}] returncode={result.returncode}; log={log_path}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    return result

common = ['--repo', str(REPO), '--dataset', str(DATASET), '--workspace', str(WORKSPACE)]
subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'resolve-base', '--repo', str(REPO), '--workspace', str(WORKSPACE)], cwd=REPO, check=True)
subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'preflight', *common, '--base-model', str(BASE_MODEL), '--required-gpu', 'A100'], cwd=REPO, check=True)
gate_log = WORKSPACE / 'gates_command.log'
gate_result = run_logged('gates', [str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'gates', *common, '--base-model', str(BASE_MODEL), '--required-gpu', 'A100'], gate_log)
if gate_result.returncode:
    gate_report = WORKSPACE / 'gates' / 'gate_report.json'
    if gate_report.is_file():
        print('PARTIAL GATE REPORT:', gate_report.read_text(encoding='utf-8'), file=sys.stderr)
    raise RuntimeError(f'GPU GATE COMMAND FAILED; full log: {gate_log}')
print('TINY TRAIN / RESUME / SPEED GATES PASS')

In [ ]:
train_log = WORKSPACE / 'train_all_command.log'
run_streaming_command([str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'train-all', *common, '--base-model', str(BASE_MODEL)], cwd=REPO, log_path=train_log, label='train-all')
print('ALL SIX HASH-LOCKED RUNS COMPLETE')

In [ ]:
evaluation_log = WORKSPACE / 'final_evaluation_command.log'
evaluation_result = run_logged('final evaluation', [str(VENV_PYTHON), '-m', 'pcb_defect.final_evaluation', *common], evaluation_log)
if evaluation_result.returncode:
    raise RuntimeError(f'FINAL EVALUATION COMMAND FAILED; full log: {evaluation_log}')
deployment_log = WORKSPACE / 'deployment_command.log'
DEPLOYMENT_RUNTIME_BEFORE = runtime_contract_state('DEPLOYMENT RUNTIME BEFORE')
deployment_result = run_logged('deployment', [str(VENV_PYTHON), '-m', 'pcb_defect.deployment', *common], deployment_log)
DEPLOYMENT_RUNTIME_AFTER = runtime_contract_state('DEPLOYMENT RUNTIME AFTER')
if DEPLOYMENT_RUNTIME_AFTER != DEPLOYMENT_RUNTIME_BEFORE:
    raise RuntimeError('ONNX Runtime state changed across the deployment command')
if deployment_result.returncode:
    for evidence in (WORKSPACE / 'deployment' / 'deployment_gate.json', WORKSPACE / 'deployment' / 'model_contract.candidate.json'):
        if evidence.is_file():
            print(f'DEPLOYMENT EVIDENCE {evidence.name}:', evidence.read_text(encoding='utf-8'), file=sys.stderr)
    raise RuntimeError(f'DEPLOYMENT COMMAND FAILED; full log: {deployment_log}')
def package_is_verified(package: Path) -> bool:
    sidecar = package.with_suffix(package.suffix + '.sha256')
    if package.exists() != sidecar.exists():
        raise RuntimeError('Partial result package exists; inspect it instead of overwriting')
    if not package.exists():
        return False
    fields = sidecar.read_text(encoding='ascii').split()
    if len(fields) != 2 or fields[1] != package.name or fields[0] != sha256_file(package):
        raise RuntimeError('Existing result package fails its SHA-256 sidecar')
    return True
package = DRIVE_ROOT / 'packages' / f'paired-results-a100-{EXPECTED_GIT_SHA[:12]}.zip'
if not package_is_verified(package):
    package_log = WORKSPACE / 'result_package_command.log'
    package_result = run_logged('result package', [str(VENV_PYTHON), '-m', 'pcb_defect.result_package', '--workspace', str(WORKSPACE), '--output', str(package)], package_log)
    if package_result.returncode:
        raise RuntimeError(f'RESULT PACKAGE COMMAND FAILED; full log: {package_log}')
if not package_is_verified(package):
    raise RuntimeError('Result package was not created atomically')
print('A100 HANDOFF COMPLETE', package, sha256_file(package))